# HACS208c COMPAS
This notebook provides a template to train your own recidivism risk prediction model.

## Background
We will use the parsed FOIA'd dataset from the analysis completed by ProPublica exposing bias in the COMPAS recidivism risk analysis model. The results of the original analysis are published in this news article: https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing. As an example of the impact of undefined and non-transparent definitions of fairness and bias, Northpointe (creators of COMPAS) has their rebuttal available here: https://go.volarisgroup.com/rs/430-MBX-989/images/ProPublica_Commentary_Final_070616.pdf (It is still available via the Wayback Machine). 

## Instructions
The original data has been split into a `train.csv` and `test.csv`. Only use the `train.csv` during model training. 

Play around with __feature engineering, model selection, and objective functions__ to try to build a __harmless__ model when predicting whether or not an individual will commit `is_violent_recid`

Compare your results with the original COMPAS model

Be prepared to share your findings on how unbiased/biased harmless/harmful your model is and how you arrived at model decisions. 

## Steps are as follows:
1. Setup and basic data cleaning (I generally do not recommend changing this)
1. Feature Engineering
1. Model and Objective Selection
1. Model Training
1. Evaluate on Test Data
    - (Optional) Compare via Model Facts Label

## Important Columns
- `v_score_text`: The COMPAS model score label (Low, Medium, High) from their violent recidivism prediction model
- `v_decile_score`: The COMPAS quantitative score (1-10)
- `is_violent_recid`: Whether or not they commited a new violent crime within two years after release (0 = not a recidivist)
- `priors_count`: The number of prior crimes. For the sake of the Model Facts Label, we will use this as a baseline for comparison

# 1. Setup
- Install necessary packages and load necessary functions
- Download the `train.csv` dataset 
- Clean data per Propublica methodology

##  Data Cleaning
- Filters out bad data:
    - Some of the data is missing scores or have start dates before end dates, demonstrating potential errors in the data. This is in line with the Propublica team's method in https://github.com/propublica/compas-analysis/blob/master/Compas%20Analysis.ipynb
- For comparison with the original model: Convert COMPAS labels (for accuracy calculations): we label all data with "High" COMPAS labels to be a prediction of violent recidivism. "Medium" and "Low" are set to 0. 
    - The Propublica team labeled High and Medium as 1. However, the Northpointe team took issue with this in their rebuttal
    - Saved in `COMPAS_pred` column
- Create a baseline prediction: While no baseline model is available, we create our own baseline under the assumption that people with a greater criminal history (`prior_counts`) are more likely to be a violent recidivist. 
    - We establish a threshold to mirror the percentage of violent recidivists in the data. 
    - This baseline should not be used in real life. It propagates the systemic biases caused by racism and overpolicing of Black and Brown neighborhoods.
    - Binary predictions saved to `BASELINE_pred` column; Probabilities saved in `BASELINE_proba`
- Format data for easy Model Facts label generation 

In [ ]:
# Download packages 
!pip install modelfacts, ydata-profiling, 'setuptools<=80.10.2'

In [ ]:
# Import packages and some initial prep functions (DO NOT CHANGE unless you know what you are doing)
import os
import requests
import pandas as pd
import datetime as dt
from sklearn import metrics

from modelfacts import  ModelFacts


def download_data(url: str, save_path: str) -> None:
    """
    Downloads a file from the specified URL and saves it to the given path.
    
    Args:
        url (str): The URL of the file to download.
        save_path (str): The local path where the file should be saved.
    """
    print(f"Downloading file from {url}...")
    response = requests.get(url)

    if response.status_code == 200:
        # Ensure the directory exists
        os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
        
        # Write the downloaded content
        with open(save_path, 'wb') as f:
            f.write(response.content)
        print(f"Successfully saved to: {save_path}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")

def prep_data(data: pd.DataFrame) -> pd.DataFrame:
    """
    Prepares the data by filtering and creating baseline and prediction columns.
    
    Args:
        data (pd.DataFrame): The input DataFrame to prepare.
    """
    # get rid of missing COMPAS values and invalid rows
    data = data[(~data['score_text'].isna()) & (data['end']>data['start'])].copy()
    # create baseline for comparison based on priors count 
    # the cutoff is the linear quantile of the percent of recidivists
    split = {'High': 1, 'Medium':0, 'Low':0}
    perc_recid = sum(data['is_violent_recid'])/len(data)
    cutoff = data['priors_count'].quantile(q=1-perc_recid)
    baseline_text = data['priors_count'].where(data['priors_count']>0, 'Low')\
        .where((data['priors_count']<=0) | (data['priors_count']>cutoff), 'Medium')\
        .where(data['priors_count']<=cutoff, 'High')

    data['BASELINE_proba'] = data['priors_count']
    data['BASELINE_pred'] = baseline_text.replace(split).astype(int)
    # create prediction column based on COMPAS score_text
    data['COMPAS_pred'] = data['v_score_text'].replace(split).astype(int)   
    return data

In [ ]:
# set data path 
train_path = "new_data/train.csv"
test_path = "new_data/test.csv"

In [ ]:
# download train and test data

TRAIN_GITHUB_URL = 'https://raw.githubusercontent.com/jhzsquared/hacs208c_lecture/main/data/train.csv'
TEST_GITHUB_URL = 'https://raw.githubusercontent.com/jhzsquared/hacs208c_lecture/main/data/test.csv'

download_data(TRAIN_GITHUB_URL, train_path)
download_data(TEST_GITHUB_URL, test_path)

In [ ]:
# load in training data
train_data = prep_data(pd.read_csv(train_path))
print("Size of training data (rows, columns):", train_data.shape)
train_data.head()

# 2. Feature Engineering
Explore the data, decide what features you want to use to train predictions, and how you want to transform them

The sklearn preprocessing module should be helpful: https://scikit-learn.org/stable/api/sklearn.preprocessing.html#module-sklearn.preprocessing 

In [ ]:
# Explore the data (I like to use this package but feel free to conduct your own EDA)
from ydata_profiling import ProfileReport

profile = ProfileReport(train_data, title="Data Exploration Report")
profile

In [ ]:
train_data.info()

In [ ]:
# Choose the columns you want to include in the model.
# Set up feature engineering functions. 
# Highly recommend using sklearn ColumnTransformer and Pipeline objects 
# so everything is self contained in the model 
# Remember: only use train data for imputation and transformations
# `is_violent_recid` is the target variable, so DO NOT include it as a feature
    # `is_recid` is too closely related to the target variable, DO NOT include it as a feature


# Example of how to set up the feature engineering pipeline:

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

categorical_features = ["sex", "race"]
numeric_features = ["age"]
count_features = ["juv_fel_count", "juv_misd_count", "juv_other_count"]
columns_to_use = categorical_features + numeric_features + count_features

categorical_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
numeric_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
count_transformer = Pipeline(steps=[
    # ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ("categorical", categorical_transformer, categorical_features),
    ("numeric", numeric_transformer, numeric_features),
    ("juvenile_counts", count_transformer, count_features),
])


# 3. Model and Objective Selection
For simplicity, I recommend using the scikit-learn package's classification models 
For more information and examples: https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html 
```


In [ ]:
# Define your model:
# Example if you decided on using a logistic regression and use 5 fold CV
# Selecting best model based on F-1 during CV (otherwise logistic regression optimizes for log loss)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

scoring = 'f1'
X = train_data[columns_to_use]
y = train_data["is_violent_recid"].astype(int)

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=28)

# set up Cross validation
search = GridSearchCV(
    model_pipeline,
    param_grid={
        "classifier__C": [0.01, 0.1, 1, 10, 100], # the hyperparameters to explore for log regression
        "classifier__class_weight": [None, "balanced"],
    },
    scoring=scoring,
    cv=cv,
    refit=True,
    n_jobs=-1,
)


In [ ]:
# Train and select the best model
search.fit(X, y)
model = search.best_estimator_

print(f"Best CV {scoring}:", search.best_score_)
print("Best parameters:", search.best_params_)

# 4. Evaluate on Test Data

Obtain the predictions on the test data and evaluate it for harms

Come up with your own evaluation method, make some charts, and/or make  a Model Facts Label

In [ ]:
# load in test data
test_data = prep_data(pd.read_csv(test_path))
print("Size of test data (rows, columns):", test_data.shape)
test_data.head()


In [ ]:
# conduct prediction on the best model and save results to test_data
from sklearn.metrics import classification_report
best_model = search.best_estimator_
test_features = test_data[columns_to_use]
test_data["model_pred"] = best_model.predict(test_features)
test_data["model_proba"] = best_model.predict_proba(test_features)[:, 1]

# Look at prediction results
print(classification_report(test_data['is_violent_recid'], test_data['model_pred']))

## Create Model Facts Label
Generate a Model Facts Label based on your model's predictions. 
Update the administrative details as needed (currently set with default values)

Variables in all caps likely do not need to be adjusted, though you can set the baseline as the Northpointe model if desired. Variables in lower case do need to be updated with your model's information

In [ ]:
# Set admin details (you should not need to change these)
APPLICATION = "Predicting risk of violent recidivism using COMPAS. " \
"The target class is predicting violent recidivism"
MODEL_TYPE = "classification"
WARNINGS = "This model has been demonstrated to propagate biases by ProPublica." \
 " Its creators claim this model is unbiased, under the predictive parity paradigm using AUC." \
 " Without a clear definition of fairness, it should not be used in decision making"
SOURCE = "Data from Broward County, Florida https://github.com/propublica/compas-analysis/tree/master." \
" Model created by Northpointe"
TRAIN_DATE = max(pd.to_datetime(test_data['compas_screening_date'])) # date compas model was used
TEST_DATA_DATE = dt.datetime(2016,5,23) # propublica article publication
DATA_SPLIT = "80/20" # train/test split
DATA_SIZE = len(test_data)+len(train_data)
AGE_COL = "age" # which column has age information in it
DEMO_COLS = ['race', 'sex', 'age'] # which columns has additional demographic information in it
TRUE_COL = "is_violent_recid"


In [ ]:
# Set your model specific information and create a Model Facts Label

# Update with your model's prediction columns if different
pred_col = "model_pred"
pred_proba = "model_proba"

# Baseline your model against these predictions
# could also be "BASELINE_pred" if you wanted to baseline it only against previous recidivism history
baseline_col = "COMPAS_pred" 
baseline_proba = "COMPAS_proba"

# Define scoring functions name (must match an sklearn.metrics function)
# See: https://scikit-learn.org/stable/api/sklearn.metrics.html#module-sklearn.metrics 
st_score = 'f1_score' # name of the scoring function that should be applied to this kind of data
st_kwargs = {} # e.g. {'average':'macro'} any of the keyword arguments for the score of interest
t_score = 'log_loss' # name of the scoring function used for training or CV optimization
t_kwargs = {} # same usage as st kwargs

# load scoring function
t_score_func = getattr(metrics, t_score)
st_score_func = getattr(metrics, st_score)

model_facts = ModelFacts(test_data, TRUE_COL, pred_col, baseline_col, 
                         st_score_func, t_score_func, classification = True, 
                         pred_proba = pred_proba, baseline_proba = baseline_proba,
                         t_proba = False)
# calculate various stats along demographics
mf_compas = model_facts(DEMO_COLS, age_col = AGE_COL,
    train_date = TRAIN_DATE, test_data_date = TEST_DATA_DATE,
    data_size = DATA_SIZE, data_split = DATA_SPLIT,
    st_kwargs = st_kwargs, t_kwargs = t_kwargs)

# create the Model Facts label from the data
table = model_facts.make_label(mf_compas, APPLICATION, WARNINGS, SOURCE, show = True)

# Generate a Model Fact's Label based on the Original Northpointe Model Predictions
As demonstrated in Northpointe's rebuttal, the scoring metric you use can be quite effective in masking biases. 

This Model Facts Label uses AUC (as Northpointe does) and calculates the results on the test data. 

In [ ]:
# Set admin details (you should not need to change these)
APPLICATION = "Predicting risk of violent recidivism using COMPAS. " \
"The target class is predicting violent recidivism"
MODEL_TYPE = "classification"
WARNINGS = "This model has been demonstrated to propagate biases by ProPublica." \
 " Its creators claim this model is unbiased, under the predictive parity paradigm using AUC." \
 " Without a clear definition of fairness, it should not be used in decision making"
SOURCE = "Data from Broward County, Florida https://github.com/propublica/compas-analysis/tree/master." \
" Model created by Northpointe"
TRAIN_DATE = max(pd.to_datetime(test_data['compas_screening_date'])) # date compas model was used
TEST_DATA_DATE = dt.datetime(2016,5,23) # propublica article publication
DATA_SPLIT = "80/20" # train/test split
DATA_SIZE = len(test_data)+len(train_data)
AGE_COL = "age" # which column has age information in it
DEMO_COLS = ['race', 'sex', 'age'] # which columns has additional demographic information in it
TRUE_COL = "is_violent_recid"

# Define variables
baseline_col = "BASELINE_pred" # could also be "COMPAS_pred" if you wanted to baseline it against COMPAS
baseline_proba = "BASELINE_proba"

# Update with your model's prediction columns 
pred_col = "model_pred"
pred_proba = "model_proba"

t_score = 'roc_auc_score' 
st_score = 'roc_auc_score'
st_proba = True
t_proba = True
t_score_func = getattr(metrics, t_score)
st_score_func = getattr(metrics, st_score)

model_facts_NP = ModelFacts(test_data, TRUE_COL, pred_col, baseline_col, 
                         st_score_func, t_score_func, classification = True, 
                         pred_proba = pred_proba, baseline_proba = baseline_proba,
                         t_proba = False)
# calculate various stats along demographics
mf_compas = model_facts_NP(DEMO_COLS, age_col = AGE_COL,
    train_date = TRAIN_DATE, test_data_date = TEST_DATA_DATE,
    data_size = DATA_SIZE, data_split = DATA_SPLIT,
    st_kwargs = st_kwargs, t_kwargs = t_kwargs)
# create the Model Facts label from the data
table = model_facts_NP.make_label(mf_compas, APPLICATION, WARNINGS, SOURCE, show = True)

## Alternatively: Use Ipywidgets to build the label
Build Model Facts Label using a form 
You will need to save out your test data into a csv so that it can be uploaded

In [ ]:
# Save test data with predictions to a new csv file
pred_path = "new_data/test_with_predictions.csv"
test_data.to_csv(pred_path, index=False)

In [ ]:
from modelfacts import ModelFactsWidget
wm = ModelFactsWidget()
wm

In [ ]:
# Run this cell after completing the form
# Creates the label 
# You can rerun this cell after reselecting values in the widget 
# without restarting the notebook
table = wm.create_label()